In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
SHA2_BITS = 256

In [0]:
%run ./../common/utilities

In [0]:
dbutils.widgets.text("catalog", "abcgroup", "Catalog")

In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
crm_sales = f"{catalog}.{silver_schema}.crm_sales"
silver_prod = f"{catalog}.{silver_schema}.crm_products"
dim_cust = f"{catalog}.{gold_schema}.dim_customers"
dim_prod = f"{catalog}.{gold_schema}.dim_products"
dim_date = f"{catalog}.{gold_schema}.dim_date"

In [0]:
df = (
    spark.table(crm_sales).alias("sd")
    .join(
        spark.table(dim_prod).alias("pr"),
        F.col("sd.product_number") == F.col("pr.product_number"),
        "left"
    )
    .join(
        spark.table(silver_prod).alias("sp"),
        F.col("sd.product_number") == F.col("sp.product_number"),
        "left"
    )
    .join(
        spark.table(dim_date).alias("dd"),
          F.to_date(F.col("sd.order_date")) == F.col("dd.full_date"),
          "left"
    )
    .join(
        spark.table(dim_cust).alias("cu"),
        F.col("sd.customer_id") == F.col("cu.customer_id"),
        "left"
    )
    .select(
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("sd.order_number"), F.lit("")),
                F.coalesce(F.col("sd.product_number"), F.lit("")),
                F.coalesce(F.col("sd.customer_id"), F.lit(""))
            ),
            SHA2_BITS
        ).alias("sales_key"),

        F.col("sd.order_number"),
        F.col("sd.order_date"),
        F.col("sd.ship_date"),
        F.col("sd.due_date"),
        F.col("sd.sales_amount"),
        F.col("sd.quantity"),
        F.col("sd.price"),
        F.col("sp.product_cost"),

        F.col("pr.product_key"),
        F.col("cu.customer_key"),
        F.col("dd.date_key").alias("order_date_key"),

        F.col("dd.year").alias("order_year"),
        F.col("dd.month").alias("order_month"),
        F.col("dd.quarter").alias("order_quarter")
    )
)

In [0]:
display(df.limit(5))

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable(f"{catalog}.{gold_schema}.fact_sales")

In [0]:
display(spark.sql(f"""
    SELECT *
    FROM {catalog}.{gold_schema}.fact_sales
    LIMIT 5
"""))